This will try out a few multi-panel/cross panel questions to add to our QA.

In [1]:
save_dir = '~/Dropbox/jcdl_followup/synthetic_figures/' # where images are saved

In [2]:
from glob import glob
import json
import os
import numpy as np
from importlib import reload
from copy import deepcopy

from utils.plot_qa_utils import plot_index_to_words
from utils.plot_qa_utils import init_qa_pairs
from utils.figure_level_qa_utils import figure_level_qa


save_dir = os.path.expanduser(save_dir)

In [3]:
# get jsons
img_files = glob(save_dir+'jsons/*.json')
img_files[:3]

['/Users/jnaiman/Dropbox/jcdl_followup/synthetic_figures/jsons/Picture_000121.json',
 '/Users/jnaiman/Dropbox/jcdl_followup/synthetic_figures/jsons/Picture_000064.json',
 '/Users/jnaiman/Dropbox/jcdl_followup/synthetic_figures/jsons/Picture_000176.json']

In [4]:
def format_index(data):
    index_format = 'Assume the following indexing for the plots:'
    for k,v in data.items():
        if 'plot' in k:
            kind = k.split('plot')[-1]
            index_format += ' plot ' + kind + ' is the ' + plot_index_to_words(data['figure']['plot indexes'][int(kind)]) + ' panel,'
    index_format = index_format.removesuffix(',') + '.'
    return index_format

In [5]:
def calc_strongest(data, verbose=True):
    relations = {'plots':[], 'avalues':[]}

    for k,v in data.items():
        if 'plot' in k:
            if data[k]['distribution'] == 'linear' and data[k]['type'] != 'contour' and data[k]['type'] != 'histogram':
                relations['plots'].append(k)
                try:
                    relations['avalues'].append(data[k]['data']['data params']['points']['a'])
                except:
                    try:
                        relations['avalues'].append(data[k]['data']['data params']['a'])
                    except:
                        if 'line0' in data[k]['data']['data params']: # multiline plot
                            if verbose:
                                print('Multi-line plot with more than 1 linear relationship -- taking max')
                                aa = []
                                for lk, l in data[k]['data']['data params'].items():
                                    aa.append(l['a'])
                                i = np.argmax(np.abs(aa))
                                relations['avalues'].append(aa[i])
                        else:
                            print(k,v)
                            lskj

    max_relation = np.argmax(np.abs(relations['avalues']))
    plot_strongest = 'Plot ' + str(relations['plots'][max_relation].split('plot')[-1])
    if verbose:
        print('all relations:')
        for p,a in zip(relations['plots'],relations['avalues']):
            print('  ', p, ', a =', a, ', type =', data[p]['type'])
        print('')
        print('Plot with strongest linear relationship:')
        print('  ', plot_strongest)
    return plot_strongest

In [6]:

for imgf in img_files:
    hasMultiLine = False
    with open(imgf,'r') as f:
        data = json.load(f)
        data = json.loads(data)
    nr = data['figure']['nrows']
    nc = data['figure']['ncols']
    nline = 0
    if nr*nc > 1:
        #print('multipanel!')
        for k,v in data.items():
            if 'plot' in k:
                if data[k]['distribution'] == 'linear' and data[k]['type'] != 'contour' and data[k]['type'] != 'histogram':
                    nline += 1
        if nline > 1:
            print('')
            print('-------------------------')
            #print('linears!')
            print(imgf.replace('/jsons/','/imgs/').removesuffix('.json')+'.jpeg')
            ans = calc_strongest(data, verbose=True)


-------------------------
/Users/jnaiman/Dropbox/jcdl_followup/synthetic_figures/imgs/Picture_000157.jpeg
Multi-line plot with more than 1 linear relationship -- taking max
Multi-line plot with more than 1 linear relationship -- taking max
all relations:
   plot1 , a = -33.928802228170404 , type = line
   plot7 , a = 50.42302989524112 , type = line

Plot with strongest linear relationship:
   Plot 7

-------------------------
/Users/jnaiman/Dropbox/jcdl_followup/synthetic_figures/imgs/Picture_000058.jpeg
Multi-line plot with more than 1 linear relationship -- taking max
all relations:
   plot0 , a = -66.79129106156782 , type = scatter
   plot1 , a = -76.06127910838647 , type = line

Plot with strongest linear relationship:
   Plot 1

-------------------------
/Users/jnaiman/Dropbox/jcdl_followup/synthetic_figures/imgs/Picture_000059.jpeg
all relations:
   plot3 , a = -80.25602424429525 , type = scatter
   plot7 , a = 88.46794963242559 , type = scatter

Plot with strongest linear relat

## Now for comparing medians of histograms

In [24]:
import utils.cross_panel_qa_utils
reload(utils.cross_panel_qa_utils)
from utils.cross_panel_qa_utils import q_large_small_stat_hists, calc_strongest_stat_hists

In [27]:
for imgf in img_files:
    hasMultiHist = False
    with open(imgf,'r') as f:
        data = json.load(f)
        data = json.loads(data)
    nr = data['figure']['nrows']
    nc = data['figure']['ncols']
    nline = 0
    plots_hist = []
    if nr*nc > 1:
        for k,v in data.items():
            if 'plot' in k:
                if data[k]['type'] == 'histogram':
                    nline += 1
                    plots_hist.append(k)
        if nline > 1:
            print('')
            print('-------------------------')
            print(imgf.replace('/jsons/','/imgs/').removesuffix('.json')+'.jpeg')

            # overall panels
            qa_pairs = init_qa_pairs()
            qa_pairs, err = q_large_small_stat_hists(data, qa_pairs, stat=np.median, 
                            relation=np.argmax, verbose=True, use_abs=False)
            #import sys; sys.exit()



-------------------------
/Users/jnaiman/Dropbox/jcdl_followup/synthetic_figures/imgs/Picture_000009.jpeg
all medians:
   plot0 , stat = 73.60734287118774 , type = histogram
   plot1 , stat = -36.17421728970121 , type = histogram

Plot with largest median:
   Plot 0
QUESTION: You are a helpful assistant that can analyze images. Assume the following indexing for the panels in this figure: plot 0 is the top-left panel, plot 1 is the top row and second from the left panel, plot 2 is the top row and third from the left panel, plot 3 is the top row and fourth from the left panel. Which plot shows the largest median data values? Please choose from the following list of plot numbers: [0, 1]. Please format the output as a json as {"cross - largest median":""} for this figure panel, where the "cross - largest median" value should be an integer referring to the panel number which shows the largest data median.
ANSWER: {'cross - largest median (list)': 0}

-------------------------
/Users/jnaima

['plot0', 'plot1']

In [ ]:
# def calc_strongest_stat_hists(data, stat=np.median, verbose=False, 
#     relation = np.argmax):

#     relations = {'plots':[], 'mvalues':[]}

#     for k,v in data.items():
#         if 'plot' in k:
#             if data[k]['type'] == 'histogram':
#                 relations['plots'].append(k)
#                 relations['mvalues'].append(stat(data[k]['data']['xs']))

#     if len(relations['plots']) <= 1: # only 1 or less
#         return '','',True

#     if 'max' in str(relation.__name__).lower():
#         lam = 'largest'
#     elif 'min' in str(relation.__name__).lower():
#         lam = 'smallest'
#     else:
#         lam = '<UNKNOWN QUANT>'

#     max_relation = relation(np.abs(relations['mvalues']))
#     plot_strongest = 'Plot ' + str(relations['plots'][max_relation].split('plot')[-1])
#     if verbose:
#         print('all '+str(stat.__name__)+'s:')
#         for p,a in zip(relations['plots'],relations['mvalues']):
#             print('  ', p, ', stat =', a, ', type =', data[p]['type'])
#         print('')
#         print('Plot with '+lam+' '+str(stat.__name__)+':')
#         print('  ', plot_strongest)
#     return plot_strongest,relations['plots'],False   

In [ ]:
# calc_strongest_stat_hists(data, verbose=True, relation=np.argmax, stat=np.std)

all stds:
   plot0 , stat = 3.2209097796689945 , type = histogram
   plot1 , stat = 0.781571386041467 , type = histogram

Plot with largest std:
   Plot 0


('Plot 0', ['plot0', 'plot1'], False)

In [ ]:
np.median.__name__

'median'

"['x', 'plot']"

In [ ]:
# data['plot0']['data']

In [ ]:
# data['figure']